# TP1 — Étape 2 : Détection et traitement des anomalies

**Dataset Lumina & Co**

Dans cette étape, nous inspectons et traitons les problèmes identifiés dans `transactions.csv` et `customers.csv`.  
Pour chaque anomalie, nous documentons la décision prise et sa justification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Chargement des données
transactions = pd.read_csv('data/transactions.csv')
customers = pd.read_csv('data/customers.csv')

print(f"transactions.csv : {transactions.shape[0]:,} lignes × {transactions.shape[1]} colonnes")
print(f"customers.csv    : {customers.shape[0]:,} lignes × {customers.shape[1]} colonnes")

---
## 1. Anomalies dans `transactions.csv`

### 1.1 — `customer_id` manquants

**Constat** : 418 258 lignes (22,8 %) n'ont pas de `customer_id`.  
**Décision** : Ces lignes correspondent à des ventes en magasin (clients non identifiés / ventes anonymes). Elles sont inutilisables pour la segmentation client car on ne peut pas les rattacher à un profil. → **Suppression**.  
**Justification** : Conserver ces lignes fausserait les agrégations par client. On documente la perte de volume.

In [ ]:
n_before = len(transactions)
n_null_cid = transactions['customer_id'].isna().sum()
print(f"Lignes sans customer_id : {n_null_cid:,} ({100*n_null_cid/n_before:.1f}%)")

# Suppression des lignes sans customer_id
transactions = transactions.dropna(subset=['customer_id'])
transactions['customer_id'] = transactions['customer_id'].astype(int)

print(f"Après suppression : {len(transactions):,} lignes (−{n_null_cid:,})")

### 1.2 — `quantity` négatives (retours / annulations)

**Constat** : 23 314 lignes ont une quantité négative. Parmi celles-ci :
- **19 493** correspondent à des factures commençant par "C" (annulations explicites)
- **3 821** sont sur des factures normales (ajustements de stock, corrections)

**Décision** : Supprimer toutes les lignes avec `quantity ≤ 0`.  
**Justification** : Les quantités négatives représentent des annulations ou des retours. Les inclure biaiserait le chiffre d'affaires et les métriques RFM à la baisse. On les exclut pour ne garder que les ventes effectives.

In [ ]:
n_before = len(transactions)

# Détail des quantités négatives
c_invoices = transactions[transactions['invoice_id'].str.startswith('C')]
neg_qty = transactions[transactions['quantity'] < 0]
neg_not_c = transactions[(transactions['quantity'] < 0) & (~transactions['invoice_id'].str.startswith('C'))]

print(f"Quantités ≤ 0 au total       : {(transactions['quantity'] <= 0).sum():,}")
print(f"  dont factures 'C' (annulations) : {len(c_invoices):,}")
print(f"  dont qty < 0 hors 'C'           : {len(neg_not_c):,}")
print(f"  dont qty NaN                     : {transactions['quantity'].isna().sum():,}")

# Suppression
transactions = transactions[transactions['quantity'] > 0]
print(f"\nAprès suppression qty ≤ 0 : {len(transactions):,} lignes (−{n_before - len(transactions):,})")

### 1.3 — `unit_price` à zéro ou négatives

**Constat** :
- **5 lignes** avec `unit_price` < 0 : ce sont des ajustements comptables ("Adjust bad debt", code produit "B", factures commençant par "A"). Montants très élevés (−11 k à −54 k).
- **~10 674 lignes** avec `unit_price` = 0 : réparties sur de nombreux codes produits standards → probablement des échantillons gratuits, promotions ou erreurs d'export.

**Décision** :
- `unit_price < 0` → **Suppression** (écritures comptables, pas des ventes)
- `unit_price = 0` → **Suppression** (pas de valeur monétaire, fausseraient line_total et les agrégations)

**Justification** : On ne peut pas calculer de chiffre d'affaires significatif sur ces lignes. Les conserver biaiserait le panier moyen à la baisse.

In [ ]:
n_before = len(transactions)

# Détail unit_price négatives
neg_price = transactions[transactions['unit_price'] < 0]
print("unit_price < 0 :")
print(neg_price[['invoice_id', 'product_code', 'product_name', 'quantity', 'unit_price']].to_string())
print()

# Détail unit_price == 0
zero_price = transactions[transactions['unit_price'] == 0]
print(f"unit_price == 0 : {len(zero_price):,} lignes")
print("Exemples de product_code concernés :")
print(zero_price['product_code'].value_counts().head(10))

# Suppression
transactions = transactions[transactions['unit_price'] > 0]
print(f"\nAprès suppression unit_price ≤ 0 : {len(transactions):,} lignes (−{n_before - len(transactions):,})")

### 1.4 — `product_code` atypiques (non-produits)

**Constat** : ~20 914 lignes ont des codes produits non standards (ne correspondant pas au format habituel `[A-Z]?\d+[A-Z]*`). Les principaux :
| Code | Description | N | Nature |
|------|-------------|---|--------|
| `POST` | Frais de port | 2 286 | Service logistique |
| `DOT` | Dotcom postage | 1 647 | Service logistique |
| `M` | Manual / ajustement | 1 619 | Entrée manuelle |
| `D` | Discount | 374 | Remise |
| `BANK CHARGES` | Frais bancaires | 302 | Écriture comptable |
| `S` | Samples | 281 | Échantillons |
| `ADJUST` | Ajustement | 256 | Correction comptable |
| `AMAZONFEE` | Commission Amazon | 236 | Frais marketplace |
| `CRUK` | Charity (Cancer Research UK) | 214 | Donation / charity |

**Décision** : Supprimer les codes comptables/non-produits (`POST`, `DOT`, `M`, `D`, `BANK CHARGES`, `S`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`).  
**Justification** : Ces lignes ne représentent pas des achats de produits. Les inclure biaiserait l'analyse du panier et la segmentation produit. Les frais de port en particulier gonflent artificiellement le CA.

In [ ]:
n_before = len(transactions)

# Codes non-produits à exclure
non_product_codes = ['POST', 'DOT', 'M', 'D', 'BANK CHARGES', 'S', 'ADJUST', 'AMAZONFEE', 'CRUK', 'B']

# Identifier les lignes avec codes non-produits
mask_non_product = transactions['product_code'].isin(non_product_codes)
print(f"Lignes avec codes non-produits : {mask_non_product.sum():,}")
print("\nDétail :")
print(transactions.loc[mask_non_product, 'product_code'].value_counts())

# Suppression
transactions = transactions[~mask_non_product]
print(f"\nAprès suppression des non-produits : {len(transactions):,} lignes (−{n_before - len(transactions):,})")

### 1.5 — Doublons exacts

**Constat** : 34 522 lignes sont des doublons exacts (toutes colonnes identiques).  
**Décision** : **Suppression des doublons**.  
**Justification** : Des lignes parfaitement identiques (même facture, même produit, même quantité, même prix, même date) sont très probablement des erreurs d'export CRM, pas des achats distincts. Un client qui achète 2× le même article sur la même facture aurait une seule ligne avec `quantity=2`.

In [ ]:
n_before = len(transactions)
n_dupes = transactions.duplicated().sum()
print(f"Doublons exacts : {n_dupes:,}")

# Suppression des doublons
transactions = transactions.drop_duplicates()
print(f"Après dé-duplication : {len(transactions):,} lignes (−{n_before - len(transactions):,})")

### 1.6 — Calcul de `line_total` et vérification de cohérence

**Décision** : Créer la colonne `line_total = quantity × unit_price` et convertir `invoice_date` en datetime.

In [ ]:
# Conversion de la date et calcul du line_total
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'])
transactions['line_total'] = transactions['quantity'] * transactions['unit_price']

print("Vérification line_total :")
print(transactions[['quantity', 'unit_price', 'line_total']].describe().round(2).to_string())
print(f"\nNb line_total ≤ 0 : {(transactions['line_total'] <= 0).sum()}")
print(f"Nb line_total NaN : {transactions['line_total'].isna().sum()}")

---
## 2. Anomalies dans `customers.csv`

### 2.1 — Cohérence des dates (`first_purchase` ≤ `last_purchase`)

**Constat** : Aucune incohérence détectée (0 cas où `first_purchase > last_purchase`).  
**Décision** : Aucune action nécessaire. On convertit simplement les colonnes en datetime.

In [ ]:
# Conversion des dates
customers['first_purchase'] = pd.to_datetime(customers['first_purchase'])
customers['last_purchase'] = pd.to_datetime(customers['last_purchase'])

# Vérification de cohérence
incoherent_dates = customers[customers['first_purchase'] > customers['last_purchase']]
print(f"Clients avec first_purchase > last_purchase : {len(incoherent_dates)}")
print("→ Aucune incohérence détectée ✓")

### 2.2 — Valeurs aberrantes dans `total_spent`, `n_orders`, `avg_basket`

**Constat** (méthode IQR, seuil 1.5×) :
| Métrique | Seuil supérieur | Nb outliers | % du dataset |
|----------|----------------|-------------|-------------|
| `total_spent` | 645 | 5 827 | 11.7% |
| `n_orders` | 11.4 | 4 059 | 8.1% |
| `avg_basket` | 140 | 3 812 | 7.6% |

**Décision** : **Conserver** les outliers mais les **documenter et visualiser**.  
**Justification** : Dans un contexte retail, les gros acheteurs (potentiels B2B) sont des clients réels et importants. Les exclure serait une perte d'information. Ils représentent probablement les 20% de clients générant 80% du CA (loi de Pareto). On les signale pour que les analyses futures en tiennent compte (winsorisation si nécessaire pour certains modèles).

**Note** : 7 558 clients ont `n_orders < 1` (valeurs fractionnaires comme 0.85). Ceci est cohérent avec le processus de bootstrapping mentionné dans le DATA_CARD (rééchantillonnage stratifié par quintile RFM). → **Conservé tel quel**.

In [ ]:
# Détection des outliers avec la méthode IQR
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['total_spent', 'n_orders', 'avg_basket']):
    q1 = customers[col].quantile(0.25)
    q3 = customers[col].quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    n_out = (customers[col] > upper).sum()
    
    ax.boxplot(customers[col], vert=True)
    ax.set_title(f"{col}\n{n_out} outliers (> {upper:.0f})")
    ax.set_ylabel(col)

plt.suptitle("Boxplots des métriques clients — Détection des valeurs aberrantes", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Clients avec n_orders < 1
print(f"\nClients avec n_orders < 1 (fractionnaires / bootstrapping) : {(customers['n_orders'] < 1).sum():,}")
print(f"Exemple :")
print(customers[customers['n_orders'] < 1][['customer_id', 'n_orders', 'total_spent', 'avg_basket']].head(5).to_string())

### 2.3 — Clients avec une seule transaction vs clients récurrents

**Constat** : 2 235 clients (4,5%) n'ont qu'une seule commande (`n_orders ≈ 1`).  
**Décision** : **Conserver** — un client one-shot est un profil marketing valide et intéressant (cible potentielle pour des campagnes de réactivation).  
**Justification** : Exclure ces clients biaiserait la segmentation vers les clients fidèles uniquement.

In [ ]:
# Proportion clients one-shot vs récurrents
single = (customers['n_orders'].round() == 1).sum()
recurring = len(customers) - single

print(f"Clients one-shot (≈1 commande) : {single:,} ({100*single/len(customers):.1f}%)")
print(f"Clients récurrents (>1)         : {recurring:,} ({100*recurring/len(customers):.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['One-shot', 'Récurrents'], [single, recurring], color=['#e74c3c', '#2ecc71'])
ax.set_ylabel('Nombre de clients')
ax.set_title('Répartition clients one-shot vs récurrents')
for i, v in enumerate([single, recurring]):
    ax.text(i, v + 200, f"{v:,}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Récapitulatif du nettoyage `transactions.csv`

In [ ]:
# Bilan final des transactions nettoyées
print("=" * 60)
print("BILAN DU NETTOYAGE — transactions.csv")
print("=" * 60)
print(f"Lignes initiales         : 1,837,137")
print(f"Lignes après nettoyage   : {len(transactions):,}")
print(f"Lignes supprimées        : {1_837_137 - len(transactions):,} ({100*(1_837_137 - len(transactions))/1_837_137:.1f}%)")
print()
print("Colonnes :", list(transactions.columns))
print()
print("Nulls restants :")
print(transactions.isnull().sum())
print()
print("Statistiques descriptives :")
print(transactions.describe().round(2).to_string())
print()
print("=" * 60)
print("BILAN — customers.csv")
print("=" * 60)
print(f"Lignes : {len(customers):,} (aucune suppression)")
print("Aucune anomalie critique détectée.")
print("Outliers documentés mais conservés (business-relevant).")